In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QUICK-PDE: Una Qiskit Function de ColibriTD
*Consulta la [referencia de la API](https://docs.quantum.ibm.com/api/functions/colibritd-pde)*

> **Note:** Las Qiskit Functions son una funcionalidad experimental disponible para los usuarios de los planes IBM Quantum&reg; Premium, Flex y On-Prem (mediante la API de IBM Quantum Platform). Se encuentran en estado de versión preliminar y están sujetas a cambios.
## Descripción general
El solucionador de Ecuaciones Diferenciales Parciales (PDE) que se presenta aquí forma parte de nuestra plataforma Quantum Innovative Computing Kit (QUICK) (QUICK-PDE) y está empaquetado como una Qiskit Function. Con la función QUICK-PDE puedes resolver ecuaciones diferenciales parciales específicas de dominio en QPUs de IBM Quantum. Esta función se basa en el algoritmo descrito en [el artículo de descripción del H-DES de ColibriTD](https://arxiv.org/abs/2410.01130). Este algoritmo puede resolver problemas complejos de múltiples físicas, comenzando con Dinámica de Fluidos Computacional (CFD) y Deformación de Materiales (MD), con más casos de uso próximamente.

Para abordar las ecuaciones diferenciales, las soluciones de prueba se codifican como combinaciones lineales de funciones ortogonales (típicamente polinomios de Chebyshev, y más específicamente $2^n$ de ellos, donde $n$ es el número de qubits que codifican tu función), parametrizadas por los ángulos de un Circuito Cuántico Variable (VQC). El ansatz genera un estado que codifica la función, que se evalúa mediante observables cuyas combinaciones permiten evaluar la función en todos los puntos. Luego puedes evaluar la función de pérdida en la que se codifican las ecuaciones diferenciales y ajustar los ángulos en un bucle híbrido, como se muestra a continuación. Las soluciones de prueba se acercan gradualmente a las soluciones reales hasta alcanzar un resultado satisfactorio.

![Flujo de trabajo de la función QUICK-PDE](../docs/images/guides/colibritd-equation-solver/diagram.svg)

Además de este bucle híbrido, también puedes encadenar distintos optimizadores. Esto es útil cuando quieres que un optimizador global encuentre un buen conjunto de ángulos y luego un optimizador más fino siga un gradiente hasta el mejor conjunto de ángulos vecinos. En el caso de la dinámica de fluidos computacional (CFD), la secuencia de optimización predeterminada produce los mejores resultados; en el caso de la deformación de materiales (MD), aunque la opción predeterminada proporciona buenos resultados, puedes configurarla con mayor detalle para obtener beneficios específicos del problema.

Ten en cuenta que para cada variable de la función especificamos el número de qubits (con el que puedes experimentar). Al apilar 10 circuitos idénticos y evaluar los 10 observables idénticos en diferentes qubits a lo largo de un gran circuito, puedes aplicar mitigación de ruido dentro del proceso de optimización CMA, apoyándote en el método del noise learner, y reducir significativamente el número de shots necesarios.

### Dinámica de fluidos computacional
La ecuación de Burgers no viscosa modela fluidos no viscosos en movimiento de la siguiente manera:

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0,$$

$u$ representa el campo de velocidad del fluido. Este caso de uso tiene una condición de frontera temporal: puedes seleccionar la condición inicial y luego permitir que el sistema se relaje. Actualmente, las únicas condiciones iniciales aceptadas son funciones lineales: $ax + b$.

Los argumentos para las ecuaciones diferenciales de CFD se encuentran en una cuadrícula fija, de la siguiente manera:

- $t$ está entre 0 y 0.95 con 30 puntos de muestreo. $x$ está entre 0 y 0.95 con un tamaño de paso de 0.2375.

### Deformación de materiales
Este caso de uso se centra en la deformación hipoelástica con el ensayo de tracción unidimensional, en el que una barra fija en el espacio es jalada por su otro extremo. Describimos el problema de la siguiente manera:

$$u' - \frac{\sigma}{3K} - \frac{2}{\sqrt{3}}\epsilon_0\left(\frac{\sigma'}{\sigma_0\sqrt{3}}\right)^n = 0$$

$$\sigma' - b = 0,$$

$K$ representa el módulo volumétrico del material que se estira, $n$ el exponente de una ley de potencias, $b$ la fuerza por unidad de masa, $\epsilon_0$ el límite de tensión proporcional, $\sigma_0$ el límite de deformación proporcional, $u$ la función de tensión y $\sigma$ la función de deformación.

La barra considerada tiene longitud unitaria. Este caso de uso tiene una condición de frontera para la tensión superficial $t$, es decir, la cantidad de trabajo necesaria para estirar la barra.

Los argumentos para las ecuaciones diferenciales de MD se encuentran en una cuadrícula fija, de la siguiente manera:

- $x$ está entre 0 y 1 con un tamaño de paso de 0.04.

## Benchmarks
La siguiente tabla presenta estadísticas de varias ejecuciones de nuestra función.

| Ejemplo                             | Número de qubits | Inicialización        | Error     | Tiempo total (min) | Uso de runtime (min) |
| ----------------------------------- | ---------------- | --------------------- | --------- | ------------------ | -------------------- |
| Ecuación de Burgers no viscosa      | 50               | `PHYSICALLY_INFORMED` | $10^{-2}$ | 66                 | 25                   |
| Ensayo de tracción hipoelástica 1D  | 18               | `RANDOM`              | $10^{-2}$ | 123                | 100                  |

## Comenzar
Rellena el [formulario para solicitar acceso a la función QUICK-PDE](https://forms.cloud.microsoft/e/3Wi9cbjQPK). Luego, suponiendo que ya has [guardado tu cuenta](/guides/functions#install-qiskit-functions-catalog-client) en tu entorno local, selecciona la función de la siguiente manera:

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

quick = catalog.load("colibritd/quick-pde")

## Ejemplos

Para empezar, prueba uno de los siguientes ejemplos:

### Dinámica de Fluidos Computacional (CFD)

Cuando las condiciones iniciales se establecen como $u(0,x) = x$, los resultados son los siguientes:

In [ ]:
# launch the simulation with initial conditions u(0,x) = a*x + b
job = quick.run(use_case="cfd", physical_parameters={"a": 1.0, "b": 0.0})

Verifica el [estado](/guides/functions#check-job-status) de la carga de trabajo de tu Qiskit Function u obtén los [resultados](/guides/functions#retrieve-results) de la siguiente manera:

In [ ]:
print(job.status())
solution = job.result()

'QUEUED'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

_ = plt.figure()
ax = plt.axes(projection="3d")

# plot the solution using the 3d plotting capabilities of pyplot
t, x = np.meshgrid(solution["samples"]["t"], solution["samples"]["x"])
ax.plot_surface(
    t,
    x,
    solution["functions"]["u"],
    edgecolor="royalblue",
    lw=0.25,
    rstride=26,
    cstride=26,
    alpha=0.3,
)
ax.scatter(t, x, solution["functions"]["u"], marker=".")
ax.set(xlabel="t", ylabel="x", zlabel="u(t,x)")

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif" alt="Output of the previous code cell" />

### Material deformation

The material deformation use case requires the physical parameters of your material and the applied force, as follows:

In [ ]:
import matplotlib.pyplot as plt

# select the properties of your material
job = quick.run(
    use_case="md",
    physical_parameters={
        "t": 12.0,
        "K": 100.0,
        "n": 4.0,
        "b": 10.0,
        "epsilon_0": 0.1,
        "sigma_0": 5.0,
    },
)

# plot the result
solution = job.result()

_ = plt.figure()
stress_plot = plt.subplot(211)
plt.plot(solution["samples"]["x"], solution["functions"]["u"])
strain_plot = plt.subplot(212)
plt.plot(solution["samples"]["x"], solution["functions"]["sigma"])

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif" alt="Output of the previous code cell" />

![Resultado de la celda de código anterior](../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif)

### Deformación de materiales
El caso de uso de deformación de materiales requiere los parámetros físicos del material y la fuerza aplicada, de la siguiente manera:

In [ ]:
# u(t=0.2, x=0.7) == 2
assert solution["samples"]["t"][1] == 0.2
assert solution["samples"]["x"][2] == 0.7
assert solution["functions"]["u"][1, 2] == 2

![Resultado de la celda de código anterior](../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif)

El siguiente es un ejemplo de cómo obtener el valor de la función para un conjunto específico de coordenadas:

In [ ]:
job = quick.run(use_case="mdf", physical_params={})

print(job.error_message())


# or write a wrapper around it for a more human readable version
def pprint_error(job):
    print("".join(eval(job.error_message())["error"]))


print("___")
pprint_error(job)

{"error": ["qiskit.exceptions.QiskitError: 'Unknown argument \"physical_params\", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'\n"]}
___
qiskit.exceptions.QiskitError: 'Unknown argument "physical_params", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'



## Obtener mensajes de error
Si el estado de tu carga de trabajo es `ERROR`, usa `job.error_message()` para obtener el mensaje de error que te ayude a depurar, de la siguiente manera: